# Week 3 Concepts — Multimodal Document & Image AI

A single runnable notebook covering all four days, using a second worked
scenario (a used bookstore) to complement the hardware-store examples in the
daily notes. Every cell that only needs the Python standard library plus
`bs4`, `pandas`, `pypdf`, or `jsonschema` has been executed in a real sandbox
and its printed output checked against the comments in the cell above it.

**One exception, noted once here rather than in every cell:** cells that
would call a live multimodal API (image description, image-to-JSON
extraction) cannot run in this environment — there's no network access and
no API key. Those cells are clearly marked `# NOT EXECUTED IN THIS SANDBOX`
and use the current, real Anthropic API request shapes (verified against
the SDK documentation, not guessed), so they're correct code to run
elsewhere, just not runnable here.

## Day 1: Classifying Book Cover Photos With a Pretrained Multimodal Model

A used bookstore photographs every incoming book. Instead of training a CNN
to recognize genre/condition from scratch — which would need thousands of
labeled cover photos per genre and a real training pipeline — we call a
pretrained vision-language model directly and ask it. Below is a
**mock-first** wrapper: the interface is real, the network call is stubbed
out with a deterministic fallback, so the rest of the pipeline can be built
and tested without spending money or needing network access.

In [ ]:
import os

def classify_book_photo(image_path: str, api_key: str | None = None) -> dict:
    """Classify a photo of a book (genre guess + condition guess).

    Falls back to a deterministic mock response when no API key is set,
    so this function is safe to call in a notebook with no network access,
    and so tests that call it don't need a live model to pass.
    """
    # -> str | None: explicit arg wins, otherwise fall back to the environment
    api_key = api_key or os.getenv("VISION_API_KEY")
    if not api_key:
        # -> dict with keys [genre_guess, condition_guess, source], all str;
        # fixed values so repeated calls are byte-identical (useful for tests)
        return {
            "genre_guess": "science fiction",
            "condition_guess": "good (minor shelf wear)",
            "source": "mock",
        }

    # Real call would go here: base64-encode the image, send it in an
    # `image` content block alongside a `text` block asking for genre and
    # condition, in one request to a vision-capable model. See the Day 2
    # cell below for the exact current request shape.
    raise NotImplementedError("real call not available in this sandbox")


# Run twice with no API key present, to confirm the mock path is
# deterministic — this is the property that makes mock-first useful.
os.environ.pop("VISION_API_KEY", None)
result_a = classify_book_photo("dune_1965_cover.jpg")
result_b = classify_book_photo("dune_1965_cover.jpg")
print("result:", result_a)
print("deterministic (result_a == result_b):", result_a == result_b)
assert result_a == result_b

**Cost/time contrast:** training a genre-and-condition classifier from
scratch would need thousands of labeled cover photos (bookstores don't have
that lying around) and a real GPU training run measured in hours to days.
The mock-first call above costs nothing to develop against, and a real
deployment costs one inference call per photo — no labeling, no training
loop, no retraining when a new genre shows up on the shelf.

In [ ]:
# NOT EXECUTED IN THIS SANDBOX (no network access, no API key here).
# This is the real request shape for a vision-capable call to the current
# Claude API — shown so the pattern is copy-pasteable, not because it runs
# in this notebook. See the top-of-notebook note for why.

import base64
import anthropic

def real_classify_book_photo(image_path: str) -> str:
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

    with open(image_path, "rb") as f:
        # base64.standard_b64encode -> bytes -> .decode("utf-8") -> str,
        # the exact form the API's base64 image source expects
        image_data = base64.standard_b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model="claude-opus-5",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                # image block first, then the text instruction — order matters
                # for how the model attends to the two pieces of content
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_data}},
                {"type": "text", "text": "What genre is this book, and what condition is the cover in?"},
            ],
        }],
    )
    # response.content -> list of content blocks (TextBlock, ThinkingBlock, ...);
    # always check .type before reading .text, since not every block has one
    return next(block.text for block in response.content if block.type == "text")

## Day 2: Extracting Structured JSON From a Packing Slip Photo

Incoming book shipments arrive with a paper packing slip. We prompt a
multimodal model for **structured JSON** (not a free-text description), then
defensively parse whatever text comes back with a `safe_json()` helper,
since LLM output is not guaranteed to be valid JSON even when explicitly
asked for it — and even valid JSON can be the wrong *shape*, which is what
the schema validation step after it is for.

In [ ]:
import json

def safe_json(raw_text: str, default=None):
    """Parse model output as JSON, tolerating the most common wrapper
    (a markdown code fence) models add out of habit.

    Returns `default` instead of raising if parsing fails, so a single
    bad response degrades one record instead of crashing the whole batch.
    """
    # -> str, fence markers and surrounding whitespace stripped from both ends
    cleaned = raw_text.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(cleaned)  # -> dict | list | str | int | float | bool | None
    except (json.JSONDecodeError, TypeError):
        return default


# Case 1: clean, fenced JSON — the common case when the model behaves
clean_response = '```json\n{"supplier": "Riverton Books Wholesale", "item_count": 42}\n```'
parsed_clean = safe_json(clean_response)
print("clean case ->", parsed_clean)
assert parsed_clean == {"supplier": "Riverton Books Wholesale", "item_count": 42}

# Case 2: prose wrapped around the fence — safe_json only strips fences at
# the very start/end, so leading/trailing prose defeats it. This is a real,
# tested limitation, not a hypothetical one.
prose_response = 'Sure, here you go:\n```json\n{"supplier": "Riverton Books Wholesale"}\n```\nLet me know if you need more!'
parsed_prose = safe_json(prose_response, default={"__parse_failed__": True})
print("prose-wrapped case ->", parsed_prose)
assert parsed_prose == {"__parse_failed__": True}
print("Both cases behaved as expected.")

In [ ]:
from jsonschema import Draft202012Validator

# Schema for a packing slip: strict on which keys must exist, loose on
# numeric formatting since the model may return a price as a string.
PACKING_SLIP_SCHEMA = {
    "type": "object",
    "properties": {
        "supplier": {"type": "string"},
        "ship_date": {"type": ["string", "null"]},
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "quantity": {"type": "number"},
                },
                "required": ["title", "quantity"],
            },
        },
    },
    "required": ["supplier", "items"],
}

def validate_and_repair(data: dict, schema: dict) -> tuple[dict | None, list[str]]:
    """Validate `data` against `schema`; try a small set of mechanical
    repairs for common near-misses (numeric-looking strings, missing
    optional keys), then re-validate exactly once.

    Returns (repaired_data_or_None, repair_notes). `None` means the data
    could not be made schema-valid automatically -> route to human review.
    """
    validator = Draft202012Validator(schema)
    if not list(validator.iter_errors(data)):
        return data, []  # -> already valid, nothing to repair

    notes = []
    repaired = json.loads(json.dumps(data))  # cheap deep copy

    def coerce_number(value):
        if isinstance(value, str):
            try:
                return float(value.replace(",", "").strip())
            except ValueError:
                return value
        return value

    for item in repaired.get("items", []):
        if isinstance(item.get("quantity"), str):
            before = item["quantity"]
            item["quantity"] = coerce_number(item["quantity"])
            if item["quantity"] != before:
                notes.append(f"coerced quantity {before!r} -> {item['quantity']!r}")

    if "ship_date" not in repaired:
        repaired["ship_date"] = None
        notes.append("filled missing ship_date with null")

    errors_after = list(validator.iter_errors(repaired))
    if errors_after:
        return None, notes + [f"unrepairable: {e.message}" for e in errors_after]
    return repaired, notes


# Malformed model output: quantity as a string, ship_date missing entirely
malformed = {
    "supplier": "Riverton Books Wholesale",
    "items": [{"title": "Dune (1965, 1st ed.)", "quantity": "3"}],
}
repaired, notes = validate_and_repair(malformed, PACKING_SLIP_SCHEMA)
print("repaired:", repaired)
print("notes:", notes)
assert repaired is not None
assert repaired["items"][0]["quantity"] == 3.0
assert repaired["ship_date"] is None

# Genuinely broken: missing the required "items" key -> correctly gives up
broken = {"supplier": "Riverton Books Wholesale"}
repaired_broken, notes_broken = validate_and_repair(broken, PACKING_SLIP_SCHEMA)
print("\nunrepairable case ->", repaired_broken, notes_broken)
assert repaired_broken is None

**Object detection contrast:** a YOLO-style detector could count how many
boxes sit on a loading-dock photo — its output is a list of
`(class, confidence, x, y, width, height)` tuples, bounding boxes with
labels. It has no concept of "supplier name" or "ship date"; that requires
prompted, semantic extraction like `validate_and_repair` above, not object
detection. The two are complementary: detect-and-count for the boxes,
prompted extraction for the paperwork that came with them.

**PII note:** a packing slip can carry a name, phone number, or partial
account number. Mask what you don't need to keep in reversible form before
it's stored — see the `mask_card_number`-style helper on Day 2 of the daily
notes for the exact pattern; the same masking logic applies to any
digit-sequence PII, not just card numbers.

In [ ]:
import re

def mask_digits(raw: str, keep_last: int = 4) -> str:
    """Mask all but the last `keep_last` digits found in a string of text.

    General-purpose version of the card-number masker from the daily
    notes — works for phone numbers, account numbers, or any digit
    sequence that shouldn't be stored or logged in full.
    """
    digits_only = re.sub(r"\D", "", raw)  # -> str, digits only, e.g. "5551234567"
    if len(digits_only) <= keep_last:
        return "*" * len(digits_only)
    return "*" * (len(digits_only) - keep_last) + digits_only[-keep_last:]

print(mask_digits("call (555) 123-4567"))   # phone number
print(mask_digits("acct #90048812234"))     # account number
assert mask_digits("call (555) 123-4567") == "******4567"
assert mask_digits("acct #90048812234") == "*******2234"

In [ ]:
# NOT EXECUTED IN THIS SANDBOX (no network access, no API key here).
# The modern alternative to prompt-and-parse: ask the API to enforce the
# JSON shape server-side, so a malformed response literally can't happen.

from pydantic import BaseModel

class PackingSlipItem(BaseModel):
    title: str
    quantity: float

class PackingSlip(BaseModel):
    supplier: str
    ship_date: str | None
    items: list[PackingSlipItem]

# response = client.messages.parse(
#     model="claude-opus-5",
#     max_tokens=1024,
#     messages=[{"role": "user", "content": [
#         {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_data}},
#         {"type": "text", "text": "Extract the packing slip fields."},
#     ]}],
#     output_format=PackingSlip,
# )
# slip = response.parsed_output  # -> PackingSlip, already schema-valid, no safe_json needed

## Day 3: Summarizing a Multi-Page Book Condition Report

An appraiser's PDF condition report can run 10+ pages. We extract text with
`pypdf`, confirm what happens when a page has no text layer (the
scanned-PDF case), summarize long reports in map-reduce chunks, and finally
check that every number the summary cites is actually grounded in the
source text — the cheapest defense against a fabricated appraisal value.

In [ ]:
# Build a real, small PDF so pypdf extraction below runs against genuine
# PDF bytes instead of a hand-waved example. reportlab is a lightweight
# pure-Python package; if it isn't installed and there's no network access
# to fetch it, this cell prints a note and the rest of the notebook still
# works (later cells use plain strings instead of re-reading this file).
try:
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import letter

    c = canvas.Canvas("dune_first_edition_appraisal.pdf", pagesize=letter)
    c.drawString(72, 720, "Appraisal Report: Dune, 1965 Chilton Books 1st edition")
    c.drawString(72, 700, "Estimated value: 4200 USD")
    c.drawString(72, 680, "Condition: Very Good, minor spine wear, no foxing")
    c.showPage()
    c.drawString(72, 720, "Page 2: Provenance")
    c.drawString(72, 700, "Single private owner since 1971, purchase receipt included")
    c.showPage()
    c.save()
    print("wrote dune_first_edition_appraisal.pdf")
except ImportError:
    print("reportlab not available in this environment; skipping real-PDF generation")

In [ ]:
from pypdf import PdfReader

reader = PdfReader("dune_first_edition_appraisal.pdf")
# -> PdfReader with .pages; nothing is decoded until you call extract_text()
print("num pages:", len(reader.pages))

# `or ""` matters: extract_text() returns None/"" for a page with no
# recoverable text layer -- see the scanned-PDF cell right after this one
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
print("extracted text:")
print(full_text)

assert "4200" in full_text
assert "1971" in full_text
print("\nExtraction PASSED: both key numbers are present in the extracted text.")

In [ ]:
# The scanned-PDF trap: a page with no text-draw calls at all (here, just a
# filled rectangle) stands in for a rasterized scan. pypdf doesn't error --
# it just returns nothing, which is the actual danger if you don't check.
try:
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import letter

    c = canvas.Canvas("scanned_condition_photo.pdf", pagesize=letter)
    c.setFillColorRGB(0.9, 0.9, 0.9)
    c.rect(50, 50, 500, 700, fill=1, stroke=0)  # no text objects at all
    c.showPage()
    c.save()

    scanned_reader = PdfReader("scanned_condition_photo.pdf")
    scanned_text = (scanned_reader.pages[0].extract_text() or "").strip()
    print("extracted text from the scanned page:", repr(scanned_text))
    print("length:", len(scanned_text))
    assert scanned_text == ""
    print("Confirmed: pypdf silently returns empty text for an image-only page.")
    print("The fix: treat this page as an image and send it to a multimodal")
    print("model as an OCR substitute, the same model used on Day 1 and Day 2.")
except ImportError:
    print("reportlab not available in this environment; skipping scanned-PDF demo")

In [ ]:
def chunk_text(text: str, max_words: int = 500) -> list[str]:
    """Split `text` into chunks of at most `max_words` words, breaking
    only on whitespace so no word is ever split in half. This is the
    "map" half of map-reduce summarization.
    """
    words = text.split()  # -> list[str], one entry per whitespace-separated token
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


# Synthetic long document: a condition-report paragraph repeated to build
# something well past what you'd want to summarize in a single request.
paragraph = (
    "The 1965 Chilton Books first edition of Dune shows minor spine wear "
    "consistent with careful handling, no foxing on the interior pages, "
    "and an intact dust jacket with only light edge tanning. "
)
long_report = paragraph * 300  # -> str, roughly 8,700 words
print("synthetic report word count:", len(long_report.split()))

chunks = chunk_text(long_report, max_words=500)
print("num chunks:", len(chunks))
print("words in first chunk:", len(chunks[0].split()))
print("words in last chunk:", len(chunks[-1].split()))

# Round-trip check: rejoining every chunk must reproduce the original
# word sequence exactly -- confirms chunking neither drops nor duplicates text
assert " ".join(chunks).split() == long_report.split()
print("Round-trip check PASSED: chunking preserves every word, in order.")

In [ ]:
def fake_summarize(text: str, instruction: str | None = None) -> str:
    """Deterministic stand-in for a real model call, so this cell's
    output is reproducible without network access. A real implementation
    sends `text` to a summarization prompt and returns the model's answer.
    """
    first_sentence = text.strip().split(".")[0]
    return f"[summary of {len(text.split())} words] {first_sentence}."

def summarize_long_document(chunks: list[str], summarize_fn) -> str:
    # Map: each chunk is summarized independently -- this isolation is why
    # a claim split mid-sentence across two chunks can get lost or duplicated
    chunk_summaries = [summarize_fn(c) for c in chunks]  # -> list[str], len == len(chunks)
    # Reduce: the (much shorter) chunk summaries are combined and
    # summarized once more into the final result
    combined = "\n".join(chunk_summaries)
    return summarize_fn(combined, instruction="Combine these into one summary")

final_summary = summarize_long_document(chunks, fake_summarize)
print("final summary (truncated):", final_summary[:200])
assert final_summary.startswith("[summary of")

In [ ]:
import re

def numbers_in_summary_are_grounded(summary: str, source_text: str) -> bool:
    """Anti-hallucination check: every number cited in `summary` must
    appear as a literal substring somewhere in `source_text`. Deliberately
    a substring match, not semantic verification -- fast and mechanical,
    catching the single most damaging error class (fabricated numbers).
    """
    cited_numbers = re.findall(r"\d+(?:\.\d+)?", summary)  # -> list[str]
    return all(num in source_text for num in cited_numbers)

source = "Dune, 1965 first edition. Estimated value: 4200 USD. Condition: Very Good. Single private owner since 1971."
good_summary = "This 1965 first edition is valued at 4200 USD, owned by one collector since 1971."
bad_summary = "This 1965 first edition is valued at 6800 USD, owned by one collector since 1985."

print("good_summary grounded?", numbers_in_summary_are_grounded(good_summary, source))
print("bad_summary grounded? ", numbers_in_summary_are_grounded(bad_summary, source))
assert numbers_in_summary_are_grounded(good_summary, source) is True
assert numbers_in_summary_are_grounded(bad_summary, source) is False
print("\nGrounding check correctly separates a faithful summary from a fabricated one.")

## Day 4: Scraping an Auction Listings Table Into a CSV Report

A rare-book auction site publishes current listings as an HTML table. We
parse it with BeautifulSoup, load the rows into a dataframe, sort by price,
sanity-check the result, and export a CSV -- no AI model needed for any of
this. HTML tables are already structured data; this is a parsing problem,
not an understanding problem.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# Simulated page HTML (stands in for a real requests.get(...).text) -- two
# tables, like a real page would have (an unrelated "related lots" widget
# plus the real listings table), to show find() correctly picking one.
sample_html = '''
<html><body>
<table class="related-lots"><tr><th>You might also like</th></tr></table>
<table class="listings">
  <tr><th>lot</th><th>title</th><th>current_bid</th></tr>
  <tr><td>101</td><td>Dune, 1965 1st ed.</td><td>1200</td></tr>
  <tr><td>102</td><td>Foundation, 1951 1st ed.</td><td>950</td></tr>
  <tr><td>103</td><td>I, Robot, 1950 1st ed.</td><td>800</td></tr>
</table>
</body></html>
'''

soup = BeautifulSoup(sample_html, "html.parser")
# find() returns the FIRST match; filtering by class is what lets this
# correctly skip the unrelated "related-lots" table
listings_table = soup.find("table", class_="listings")
rows = listings_table.find_all("tr")  # -> list[Tag], len 4 (1 header + 3 data rows)
print("rows found (incl. header):", len(rows))

records = []
for row in rows[1:]:  # skip the header row
    cells_ = [td.get_text(strip=True) for td in row.find_all("td")]
    # -> list[str] of length 3 per row: [lot, title, current_bid]
    records.append({
        "lot": int(cells_[0]),
        "title": cells_[1],
        "current_bid": float(cells_[2]),
    })

df = pd.DataFrame(records)
# -> DataFrame, columns [lot: int64, title: object, current_bid: float64]
print(df)
print(df.dtypes)
assert len(df) == 3
assert list(df.columns) == ["lot", "title", "current_bid"]

In [ ]:
def sanity_check_report(df: pd.DataFrame, expected_min_rows: int = 1) -> list[str]:
    """Cheap structural checks before a scraped dataframe ships as a
    report -- catches the case where scraping "succeeds" (no exception)
    but the page layout changed and the data is garbage.
    """
    problems = []
    if len(df) < expected_min_rows:
        problems.append(f"only {len(df)} rows, expected >= {expected_min_rows}")
    if df["current_bid"].isna().any():
        problems.append("current_bid has missing values")
    if (df["current_bid"] <= 0).any():
        problems.append("current_bid has non-positive values")
    if df["lot"].duplicated().any():
        problems.append("duplicate lot numbers (possible double-parsed table)")
    return problems  # -> list[str], empty means "looks fine"

problems = sanity_check_report(df)
print("sanity check problems:", problems)
assert problems == []

report = df.sort_values("current_bid", ascending=False)
out_path = "auction_listings_report.csv"
report.to_csv(out_path, index=False)

with open(out_path) as f:
    written_csv = f.read()
print("\nwritten CSV:")
print(written_csv)
assert written_csv.splitlines()[1].startswith("101")  # highest bid (1200) sorted first

In [ ]:
from urllib.robotparser import RobotFileParser

# Parsed from a literal string here (no network call) to show the mechanics
# of robots.txt checking without depending on a real site being reachable.
robots_txt = '''
User-agent: *
Disallow: /internal-notes/
Allow: /listings/
'''
rp = RobotFileParser()
rp.parse(robots_txt.strip().splitlines())
print("can fetch /listings/rare-books ?", rp.can_fetch("*", "/listings/rare-books"))
print("can fetch /internal-notes/bids ?", rp.can_fetch("*", "/internal-notes/bids"))
assert rp.can_fetch("*", "/listings/rare-books") is True
assert rp.can_fetch("*", "/internal-notes/bids") is False

# Batch error handling: one URL "fails" (simulated) without stopping the batch
def scrape_listing_page(url: str) -> dict:
    if "broken" in url:
        raise ValueError("no <table class='listings'> found on page")
    return {"url": url, "lots_found": 3}

auction_urls = [
    "https://example-auction.test/listings/rare-books",
    "https://example-auction.test/listings/broken-page",
    "https://example-auction.test/listings/first-editions",
]
results, failures = [], []
for url in auction_urls:
    try:
        results.append(scrape_listing_page(url))
    except Exception as exc:
        failures.append({"url": url, "error": str(exc)})

print(f"\n{len(results)} succeeded, {len(failures)} failed")
print("failures:", failures)
assert len(results) == 2 and len(failures) == 1

## Summary

The same discipline shows up in a different form on every day this week:
build the mock/deterministic path first (Day 1), never trust model output
without validating its shape (Day 2), never trust a summary's numbers
without grounding them in the source (Day 3), and never trust a scrape's
success without sanity-checking its output (Day 4). The specific check
changes with the data shape -- schema validation, number grounding,
dataframe sanity checks -- but the underlying habit is the same one repeated
four times against four different failure modes.